# Using the LimSim Behavior Model

This notebook shows the public LimSim behavior-model API on a small scene built entirely in memory. It does not require WOMD files, parser setup, rolling runners, evaluation helpers, or direct access to the internal planner modules.

The user-facing flow is simple: create or load Tactics2D participants and a map, construct `LimSimBehaviorModel`, call `predict(...)`, and use the returned `Trajectory` objects in the rest of your simulation.

## 1. Setup

The notebook resolves the local repository first so it can run from either the docs folder or the repository root.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "tactics2d" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Cannot find the Tactics2D repository root.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
from shapely.geometry import LineString

from tactics2d.behavior import LimSimBehaviorModel, LimSimConfig
from tactics2d.map.element import Lane, LaneRelationship, Map
from tactics2d.participant.element import Vehicle
from tactics2d.participant.trajectory import State, Trajectory

print("repo root:", repo_root)

## 2. Construct The Behavior Model

`LimSimConfig` controls the prediction horizon and the search budget. The values below keep the demo fast while still producing multi-step trajectories.

In [ ]:
config = LimSimConfig(
    horizon_steps=20,
    dt=0.1,
    mcts_iterations=40,
    interaction_distance=18.0,
)
model = LimSimBehaviorModel(config)

print("planning steps:", config.planning_steps)
print("LimSim horizon_steps:", config.horizon_steps)
print("step time:", config.dt, "seconds")

## 3. Build A Minimal Road Map

The demo road has two straight lanes in the same direction. The lanes include centerlines and neighbor relationships so the behavior model can match vehicles to lanes and consider lane-change actions when they are useful.

In [ ]:
def build_lane(lane_id, x_left, x_right, y_start, y_end):
    center_x = 0.5 * (x_left + x_right)
    return Lane(
        id_=lane_id,
        left_side=LineString([(x_left, y_start), (x_left, y_end)]),
        right_side=LineString([(x_right, y_start), (x_right, y_end)]),
        speed_limit=12.0,
        speed_limit_unit="m/s",
        custom_tags={"centerline": np.array([[center_x, y_start], [center_x, y_end]])},
    )


map_ = Map(name="limsim_minimal_two_lane_road", scenario_type="demo")
lane_a = build_lane("A", 0.0, 3.6, 0.0, 90.0)
lane_b = build_lane("B", 3.6, 7.2, 0.0, 90.0)
lane_a.add_related_lane("B", LaneRelationship.RIGHT_NEIGHBOR)
lane_b.add_related_lane("A", LaneRelationship.LEFT_NEIGHBOR)
map_.add_lane(lane_a)
map_.add_lane(lane_b)

print("lanes:", list(map_.lanes))
print("map boundary:", map_.boundary)

## 4. Build Participants In Memory

Both behavior-model demos use the normal Tactics2D scene format: `participants` is a dictionary of `Vehicle` objects, each vehicle owns a `Trajectory`, and each trajectory contains time-indexed `State` objects.

For this public LimSim `predict(...)` demo, each active vehicle only needs a state at the current `frame`. Each state should provide `frame`, `x`, `y`, `heading`, and either `vx`/`vy` or `speed`; vehicle `length` and `width` are used by collision checks and trajectory planning. The model returns `planning_steps` future states (`horizon_steps` in the LimSim planner config).

Rolling LimSim can also reuse previous planned trajectories as prediction memory, but that memory belongs to the rolling workflow rather than to the raw logged-history input shown here.


In [ ]:
def make_vehicle(agent_id, x, y, speed, heading=np.pi / 2, frame=0):
    trajectory = Trajectory(id_=agent_id, fps=10, stable_freq=True)
    trajectory.add_state(
        State(
            frame=frame,
            x=x,
            y=y,
            heading=heading,
            vx=speed * np.cos(heading),
            vy=speed * np.sin(heading),
        )
    )
    return Vehicle(agent_id, "vehicle", trajectory=trajectory, length=4.5, width=1.8)


participants = {
    "ego": make_vehicle("ego", x=1.8, y=8.0, speed=7.0),
    "lead": make_vehicle("lead", x=1.8, y=18.0, speed=3.5),
    "right_lane": make_vehicle("right_lane", x=5.4, y=26.0, speed=6.0),
}

print("current frame:", 0)
print("states required per participant:", 1)
print("future states returned:", config.planning_steps)

for agent_id, participant in participants.items():
    state = participant.trajectory.get_state(0)
    print(agent_id, {"x": state.x, "y": state.y, "speed": round(state.speed, 2)})

## 5. Predict Trajectories

`predict(...)` is the shared public behavior-model entry point. It returns a dictionary from participant id to future `Trajectory`.

In [ ]:
predicted = model.predict(
    participants=participants,
    map_=map_,
    frame=0,
    agent_ids=["ego", "lead", "right_lane"],
)

print("predicted agents:", list(predicted))
for agent_id, trajectory in predicted.items():
    first = trajectory.get_state(trajectory.frames[0])
    last = trajectory.get_state(trajectory.frames[-1])
    print(
        agent_id,
        "states:", len(trajectory.frames),
        "first:", (round(first.x, 2), round(first.y, 2)),
        "last:", (round(last.x, 2), round(last.y, 2)),
    )

## 6. Execute One Closed-Loop Update

After a single `predict(...)` call, a closed-loop simulation usually commits only the first future state. The participant is updated to that state, and the model can be called again at the new frame.


In [ ]:
planned_trajectories = model.predict(participants, map_, frame=0, agent_ids=["ego"])
ego_plan = planned_trajectories["ego"]

live_vehicle = make_vehicle("ego_live", x=1.8, y=8.0, speed=7.0)
next_frame = ego_plan.frames[0]
next_state = ego_plan.get_state(next_frame)
live_vehicle.add_state(next_state)

print("executed frame:", live_vehicle.current_state.frame)
print("executed location:", tuple(round(v, 2) for v in live_vehicle.current_state.location))
print("available future frames from this plan:", ego_plan.frames[:3], "...", ego_plan.frames[-3:])


## 7. Run A Short Closed-Loop Rollout

The loop below repeats the same receding-horizon pattern for a few control updates. At each update, the model predicts from the participant's latest state, the simulation commits only the first future state, and the next iteration replans from that updated state.


In [ ]:
closed_loop_participants = {
    "ego": make_vehicle("ego", x=1.8, y=8.0, speed=7.0),
    "lead": make_vehicle("lead", x=1.8, y=18.0, speed=3.5),
    "right_lane": make_vehicle("right_lane", x=5.4, y=26.0, speed=6.0),
}
controlled_ids = ["ego"]
current_frame = 0
executed_trace = [closed_loop_participants["ego"].current_state.location]

for step in range(5):
    step_predictions = model.predict(
        closed_loop_participants,
        map_,
        frame=current_frame,
        agent_ids=controlled_ids,
    )
    ego_step_plan = step_predictions["ego"]
    next_frame = ego_step_plan.frames[0]
    next_state = ego_step_plan.get_state(next_frame)
    closed_loop_participants["ego"].add_state(next_state)
    current_frame = next_frame
    executed_trace.append(next_state.location)
    print(
        f"step {step + 1}",
        "frame:", current_frame,
        "location:", tuple(round(v, 2) for v in next_state.location),
    )

print("executed trace:", [tuple(round(v, 2) for v in point) for point in executed_trace])


## 8. Use Parsed Dataset Data

The same public API works when participants and maps come from a dataset parser. This cell is intentionally not executed here because parser files are environment-specific.

In [ ]:
# from tactics2d.behavior import LimSimBehaviorModel, LimSimConfig
# from tactics2d.dataset_parser import WOMDParser
#
# parser = WOMDParser()
# participants, _ = parser.parse_trajectory(
#     scenario_index,
#     file="your_womd_file.tfrecord",
#     folder="/path/to/womd",
# )
# map_ = parser.parse_map(
#     scenario_index,
#     file="your_womd_file.tfrecord",
#     folder="/path/to/womd",
# )
#
# model = LimSimBehaviorModel(LimSimConfig())
# trajectories = model.predict(participants, map_, frame=current_frame, agent_ids=controlled_ids)

## 9. What Stays Internal

The tutorial uses only the public behavior-model entry point. Internals such as RoI selectors, rolling runners, evaluation helpers, prediction modules, MCTS nodes, and Frenet candidate samplers are kept out of the user path. They are useful for reproduction studies and visual diagnostics, which belong in example notebooks rather than the basic tutorial.